# 🚀 Tool Upload Video lên YouTube từ Google Drive
Công cụ này giúp bạn chuyển trực tiếp video từ Google Drive sang YouTube cực nhanh chỉ với một đường link.

**Hướng dẫn:**
1. Chạy từng ô (cell) từ trên xuống dưới bằng cách bấm nút ▶️ bên trái mỗi ô.
2. Đảm bảo bạn đã có file `client_secrets.json` từ Google Cloud Console.

In [ ]:
!pip install --upgrade -q google-api-python-client google-auth-oauthlib google-auth-httplib2 httplib2

In [ ]:
from google.colab import files
import os

print("Vui lòng tải lên file client_secrets.json của bạn (Lấy từ Google Cloud Console)")
if os.path.exists('client_secrets.json'):
    print("✅ File client_secrets.json đã tồn tại. Nếu muốn đổi file khác, hãy xóa nó trước.")
else:
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != 'client_secrets.json':
            os.rename(fn, 'client_secrets.json')
    print("✅ Đã tải lên thành công file cấu hình API.")

In [ ]:
#@title ⚙️ Cấu hình Video

#@markdown Dán đường link Google Drive của video vào đây. (Lưu ý: Mặc định là ô trống, vui lòng điền link trước khi chạy)
LINK_GOOGLE_DRIVE = "" #@param {type:"string"}
TRANG_THAI_VIDEO = "Riêng tư (private)" #@param ["Công khai (public)", "Riêng tư (private)", "Không công khai (unlisted)"]

In [ ]:
import os
import re
import json
import io
import time
from google.colab import auth
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    elif m > 0: return f"{m}m {s}s"
    return f"{s}s"

# Bỏ qua lỗi bắt buộc HTTPS khi dùng localhost (Fix InsecureTransportError)
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

token_file = 'youtube_token.json'

# 1. Xác thực YouTube API theo Kênh
if not os.path.exists('client_secrets.json'):
    raise FileNotFoundError("❌ Không tìm thấy file client_secrets.json! Vui lòng chạy bước tải file lên ở trên.")

youtube_credentials = None
if os.path.exists(token_file):
    youtube_credentials = Credentials.from_authorized_user_file(token_file, ['https://www.googleapis.com/auth/youtube.upload'])

if not youtube_credentials or not youtube_credentials.valid:
    if youtube_credentials and youtube_credentials.expired and youtube_credentials.refresh_token:
        youtube_credentials.refresh(Request())
    else:
        with open('client_secrets.json', 'r') as f:
            client_config = json.load(f)
        
        redirect_uri = 'http://localhost:8080/'
        if 'installed' in client_config and 'redirect_uris' in client_config['installed']:
            redirect_uri = client_config['installed']['redirect_uris'][0]
        elif 'web' in client_config and 'redirect_uris' in client_config['web']:
            redirect_uri = client_config['web']['redirect_uris'][0]

        scopes = ['https://www.googleapis.com/auth/youtube.upload']
        flow = Flow.from_client_secrets_file('client_secrets.json', scopes=scopes, redirect_uri=redirect_uri)
        
        # ÉP GOOGLE HIỆN BẢNG CHỌN KÊNH bằng tham số prompt='consent select_account'
        auth_url, _ = flow.authorization_url(prompt='consent select_account')
        
        print(f'\n=== 🔑 HƯỚNG DẪN XÁC THỰC KÊNH YOUTUBE ===')
        print('1. Nhấn vào link sau để mở trang đăng nhập Google:')
        print(auth_url)
        print('\n2. CHỌN ĐÚNG KÊNH YOUTUBE BẠN MUỐN UP VIDEO và cấp quyền.')
        print('3. Trình duyệt sẽ báo lỗi (ví dụ: không truy cập được localhost).')
        print('4. Copy ĐƯỜNG DẪN (URL) của trang lỗi đó dán vào ô bên dưới.')
        print('====================================================\n')
        
        redirect_response = input('Dán toàn bộ URL vừa copy vào đây và nhấn Enter: ').strip()
        if redirect_response.startswith('http://'):
            redirect_response = redirect_response.replace('http://', 'https://', 1)
        
        import urllib.parse as urlparse
        from urllib.parse import parse_qs
        parsed = urlparse.urlparse(redirect_response)
        code = parse_qs(parsed.query).get('code', [None])[0]
        if code:
            flow.fetch_token(code=code)
        else:
            flow.fetch_token(authorization_response=redirect_response)
        youtube_credentials = flow.credentials
        
    # Lưu lại token để dùng cho các video sau trên kênh này
    with open(token_file, 'w') as f:
        f.write(youtube_credentials.to_json())

print(f"✅ Đã tải thông tin xác thực cho kênh.")

# 2. Xử lý link Google Drive và tải file
def get_drive_id(url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    match = re.search(r'id=([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    return url.strip()

if not LINK_GOOGLE_DRIVE.strip():
    raise ValueError("❌ BẠN CHƯA NHẬP LINK! Vui lòng dán link Google Drive vào ô cấu hình bên trên trước khi chạy.")

file_id = get_drive_id(LINK_GOOGLE_DRIVE)
if not file_id:
    raise ValueError("❌ Link Google Drive không hợp lệ! Vui lòng kiểm tra lại.")

print("⏳ Đang xử lý yêu cầu quyền truy cập Google Drive...")
auth.authenticate_user() # Hiển thị popup cấp quyền Colab (chỉ hỏi lần đầu)
drive_service = build('drive', 'v3')

try:
    file_info = drive_service.files().get(fileId=file_id, fields='name', supportsAllDrives=True).execute()
except Exception as e:
    raise Exception(f"❌ Không thể truy cập file này! Chi tiết lỗi: {e}")

file_name = file_info.get('name', 'video.mp4')
FILE_PATH = f"/content/{file_name}"

if not os.path.exists(FILE_PATH):
    print(f"\n🔥 BẮT ĐẦU KÉO FILE '{file_name}' TỪ DRIVE VỀ...")
    # Thêm acknowledgeAbuse=True để bỏ qua cảnh báo quét virus cho file quá lớn (> 10GB)
    request = drive_service.files().get_media(fileId=file_id, acknowledgeAbuse=True, supportsAllDrives=True)
    fh = io.FileIO(FILE_PATH, 'wb')
    downloader = MediaIoBaseDownload(fh, request, chunksize=1024*1024*100) # Chunk 100MB
    done = False
    start_time = time.time()
    while done is False:
        status, done = downloader.next_chunk()
        if status:
            progress = status.progress()
            elapsed = time.time() - start_time
            if progress > 0 and elapsed > 0:
                speed = status.resumable_progress / elapsed / (1024 * 1024)
                eta = (elapsed / progress) - elapsed
                print(f"\r⬇️ Đang kéo về: {int(progress * 100)}% | Tốc độ: {speed:.1f} MB/s | Còn lại: {format_time(eta)}      ", end="")
    print("\n✅ Đã kéo xong video!")
else:
    print(f"✅ Video '{file_name}' đã có sẵn trong Colab.")

# 3. Cấu hình tiêu đề mặc định
video_title = os.path.splitext(file_name)[0]
video_description = ""
video_tags = []

# 4. Thực hiện upload lên YouTube
print(f"\n🚀 BẮT ĐẦU UPLOAD LÊN YOUTUBE: {video_title}...")
youtube = build('youtube', 'v3', credentials=youtube_credentials)

privacy_map = {
    "Công khai (public)": "public",
    "Riêng tư (private)": "private",
    "Không công khai (unlisted)": "unlisted"
}
privacy_status = privacy_map.get(TRANG_THAI_VIDEO, "private")

body = {
    'snippet': {
        'title': video_title,
        'description': video_description,
        'tags': video_tags,
        'categoryId': '22'
    },
    'status': {
        'privacyStatus': privacy_status
    }
}

media = MediaFileUpload(FILE_PATH, chunksize=1024*1024*100, resumable=True) # Chunk 100MB

request = youtube.videos().insert(
    part=','.join(body.keys()),
    body=body,
    media_body=media
)

response = None
start_time = time.time()
try:
    while response is None:
        status, response = request.next_chunk()
        if status:
            progress = status.progress()
            elapsed = time.time() - start_time
            if progress > 0 and elapsed > 0:
                speed = status.resumable_progress / elapsed / (1024 * 1024)
                eta = (elapsed / progress) - elapsed
                print(f"\r⬆️ Đang tải lên YT: {int(progress * 100)}% | Tốc độ: {speed:.1f} MB/s | Còn lại: {format_time(eta)}      ", end="")
    
except HttpError as e:
    if e.resp.status == 403:
        if os.path.exists(token_file):
            os.remove(token_file)
        raise Exception("\n❌ LỖI XÁC THỰC! Token cũ đã bị xóa.\n👉 NGUYÊN NHÂN: Project của bạn chưa bật YouTube Data API v3 HOẶC bạn muốn đổi kênh khác.\n👉 CÁCH XỬ LÝ: Vui lòng CHẠY LẠI ô này để đăng nhập lại, hoặc vào Google Cloud Console bật YouTube Data API lên nhé.")
    else:
        raise e
print(f"\n\n✅ HOÀN TẤT! Video của bạn đã được tải lên YouTube.")
print(f"🔗 Đường dẫn video: https://youtu.be/{response['id']}")

# Xoá file sau khi xong để giải phóng bộ nhớ (chỉ nên xoá khi up thành công)
os.remove(FILE_PATH)